# ViT neural net with pytorch (and pytorch lightning ⚡)


* We will be training a VIT on a subset tiny image dataset


In [1]:
# import standard PyTorch modules
import pytorch_lightning as pl
import pdb
import lightning
import torchmetrics
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader

# import torchvision module to handle image manipulation
import torchvision
import torchvision.transforms as transforms

W0331 02:19:26.866000 4916 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


In [2]:
from lightning.pytorch import seed_everything

seed_everything(2026, workers=True)

Seed set to 2026


2026

In [3]:
# Check PyTorch versions
print(torch.__version__)
print(torchvision.__version__)

2.11.0+cu130
0.26.0+cu130


In [4]:
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
device

'cuda'

## Load the data and visualize it

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

train_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

val_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

# ImageFolder STANDARD
train_dataset = datasets.ImageFolder(
    root="./datasets_split2/train/texture",
    transform=train_transform
)

valid_dataset = datasets.ImageFolder(
    root="./datasets_split2/val/texture",
    transform=val_transform
)

DATA = "texture"
num_classes = 5

In [6]:
num_workers = 12

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=num_workers, persistent_workers=True, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=24, shuffle=False, num_workers=num_workers, persistent_workers=True, pin_memory=True)


In [7]:
# import numpy as np
# import matplotlib.pyplot as plt

# def show_images_grid(images):  # ← Pour GRID (batch)
#     """Affiche batch d'images normalisées"""
#     images = torchvision.utils.make_grid(images)  # [3, H*3, W*3]
#     images = images
#     images = torch.clamp(images, 0, 1)
#     npimg = images.numpy().transpose((1, 2, 0))
#     plt.figure(figsize=(8, 8))
#     plt.imshow(npimg)
#     plt.axis('off')
#     plt.show()

# # ✅ USAGE CORRECT
# train_iter = iter(train_loader)
# images, labels = next(train_iter)  # [B,3,64,64] + labels
# show_images_grid(images)  # Affiche 64 images en grille 8x8

In [8]:
# class_mapping = {
#     0: "fish",
#     1: "umbrella",
#     2: "cup",
#     3: "mountain",
#     4: "beach",
#     5: "thatched_roof",
#     6: "merrygoround",
#     7: "cabbage",
#     8: "tart",
#     9: "staircase"
# }

# np.set_printoptions(linewidth=120)
# np.array([class_mapping[labels[j].item()] for j in range(64)]).reshape(8, 8)

## A ViT architecture using pytorch-lightning

Since we will use pytorch-lightning, our class needs to inherit from pl.LightningModule

In [9]:
class ConvMLP(nn.Module):
    def __init__(self, dim, mlp_ratio=4):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim * mlp_ratio)
        self.dwconv = nn.Conv2d(dim * mlp_ratio, dim * mlp_ratio, 3, 1, 1, groups=dim * mlp_ratio)
        self.fc2 = nn.Linear(dim * mlp_ratio, dim)
        self.act = nn.GELU()

    def forward(self, x, H, W):  # H,W = forme spatiale des patches
        x = self.fc1(x)
        x = self.act(x)
        B, N, C = x.shape
        x = x.transpose(1,2).reshape(B, C, H, W)  # Remettre en 2D
        x = self.dwconv(x)
        x = x.flatten(2).transpose(1,2)
        x = self.fc2(x)
        return x

class DHVTBlock(nn.Module):
    def __init__(self, dim, num_heads=8, mlp_ratio=4):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = ConvMLP(dim, mlp_ratio)

    def forward(self, x, H, W):
        x = x + self.attn(self.norm1(x), self.norm1(x), self.norm1(x))[0]
        x = x + self.mlp(self.norm2(x), H, W)
        return x

In [10]:
import os
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
import torchmetrics
from torchmetrics.classification import MulticlassConfusionMatrix
from torch.optim.lr_scheduler import ReduceLROnPlateau


class ConvMLP(nn.Module):
    def __init__(self, dim, mlp_ratio=4, dropout=0.25):
        super().__init__()
        hidden_dim = dim * mlp_ratio
        self.fc1 = nn.Linear(dim, hidden_dim)
        self.act = nn.GELU()
        self.dwconv = nn.Conv2d(hidden_dim, hidden_dim, kernel_size=3, stride=1, padding=1, groups=hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, H, W):
        x = self.fc1(x)
        x = self.act(x)

        cls_token = x[:, :1, :]
        feat_tokens = x[:, 1:, :]

        B, N, C = feat_tokens.shape
        feat_tokens = feat_tokens.transpose(1, 2).reshape(B, C, H, W)
        feat_tokens = self.dwconv(feat_tokens)
        feat_tokens = feat_tokens.flatten(2).transpose(1, 2)

        x = torch.cat([cls_token, feat_tokens], dim=1)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x


class DHVTBlock(nn.Module):
    def __init__(self, dim, num_heads=8, mlp_ratio=4, dropout=0.25):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(
            embed_dim=dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = ConvMLP(dim, mlp_ratio=mlp_ratio, dropout=dropout)

    def forward(self, x, H, W):
        x_norm = self.norm1(x)
        attn_out, _ = self.attn(x_norm, x_norm, x_norm)
        x = x + attn_out
        x = x + self.mlp(self.norm2(x), H, W)
        return x


class VisionTransformer(pl.LightningModule):
    def __init__(self, num_classes=10, confmat_log_path="confusion_matrices.json", emb_size=192, mlp_ratio=4):
        super(VisionTransformer, self).__init__()

        self.num_classes = num_classes
        self.confmat_log_path = confmat_log_path
        self.current_train_confmat = None

        self.patch_embed = nn.Sequential(
            nn.Conv2d(3, emb_size // 4, kernel_size=3, stride=2, padding=1),
            nn.GELU(),
            nn.Conv2d(emb_size // 4, emb_size // 2, kernel_size=3, stride=2, padding=1),
            nn.GELU(),
            nn.Conv2d(emb_size // 2, emb_size, kernel_size=3, stride=2, padding=1),
            nn.GELU()
        )

        self.cls_token = nn.Parameter(torch.randn(1, 1, emb_size))
        self.pos_embed = nn.Parameter(torch.randn(1, 65, emb_size))

        self.blocks = nn.ModuleList([
            DHVTBlock(emb_size, 8, mlp_ratio) for _ in range(4)
        ])
        self.norm = nn.LayerNorm(emb_size)

        self.head = nn.Linear(in_features=emb_size, out_features=num_classes)

        self.accuracy = torchmetrics.Accuracy(task='multiclass', num_classes=num_classes)

        self.train_confmat = MulticlassConfusionMatrix(num_classes=num_classes)
        self.val_confmat = MulticlassConfusionMatrix(num_classes=num_classes)

        self.train_loss = 0.0
        self.valid_loss = 0.0

    def encoder(self, x):
        B = x.shape[0]
        patches = self.patch_embed(x).flatten(2).transpose(1, 2)
        H = W = int(patches.shape[1] ** 0.5)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, patches], dim=1)
        x = x + self.pos_embed[:, :x.shape[1], :]
        for block in self.blocks:
            x = block(x, H, W)
        x = self.norm(x)
        return x[:, 0]

    def classifier(self, x):
        out = self.head(x)
        return out

    def forward(self, x):
        features = self.encoder(x)
        logits = self.classifier(features)
        return logits

    def _step_util(self, batch, batch_idx, step_type):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)
        preds = torch.argmax(logits, dim=1)

        acc = self.accuracy(preds, y)

        self.log(f"{step_type}_loss", loss, on_step=False, on_epoch=True, prog_bar=True, logger=True)
        self.log(f"{step_type}_acc", acc, on_step=False, on_epoch=True, prog_bar=True, logger=True)

        if step_type == "train":
            self.train_confmat.update(preds, y)
            self.train_loss += loss.detach().item()
        elif step_type == "valid":
            self.val_confmat.update(preds, y)
            self.valid_loss += loss.detach().item()

        return loss

    def training_step(self, batch, batch_idx):
        return self._step_util(batch, batch_idx, "train")

    def validation_step(self, batch, batch_idx):
        return self._step_util(batch, batch_idx, "valid")

    def _get_current_lr(self):
        return self.trainer.optimizers[0].param_groups[0]["lr"]

    def _append_epoch_to_json(self, epoch, train_loss, valid_loss, train_cm, val_cm):
        entry = {
            "epoch": int(epoch),
            "train_loss": round(float(train_loss), 2),
            "valid_loss": round(float(valid_loss), 2),
            "train": train_cm.cpu().tolist(),
            "val": val_cm.cpu().tolist()
        }

        if os.path.exists(self.confmat_log_path):
            with open(self.confmat_log_path, "r", encoding="utf-8") as f:
                try:
                    data = json.load(f)
                except json.JSONDecodeError:
                    data = []
        else:
            data = []

        data.append(entry)

        with open(self.confmat_log_path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=4)

    def on_train_epoch_end(self):
        lr = self._get_current_lr()
        self.log("learning_rate", lr, on_step=False, on_epoch=True, prog_bar=True, logger=True)

        self.current_train_confmat = self.train_confmat.compute().detach().cpu()
        self.train_confmat.reset()

    def on_validation_epoch_end(self):
        val_cm = self.val_confmat.compute().detach().cpu()

        if self.current_train_confmat is not None:
            self._append_epoch_to_json(
                epoch=self.current_epoch,
                train_loss=self.train_loss,
                valid_loss=self.valid_loss,
                train_cm=self.current_train_confmat,
                val_cm=val_cm
            )

        self.val_confmat.reset()
        self.current_train_confmat = None
        self.train_loss = 0.0
        self.valid_loss = 0.0

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=1e-4, weight_decay=1e-2)
        return optimizer

**Pro tip.** When coding and debugging your architecture, don't use print statements to check the sizes of input / output tensors for instance. Use pdb (python debbuger) by writing the line
```python
import pdb; pdb.set_trace()
```
at the place where you want to check stuff

In [11]:
confmat_log_path = "confusion_matrices_" + DATA + ".json"
net = VisionTransformer(num_classes = num_classes, confmat_log_path=confmat_log_path)
print(net)

VisionTransformer(
  (patch_embed): Sequential(
    (0): Conv2d(3, 48, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (1): GELU(approximate='none')
    (2): Conv2d(48, 96, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (3): GELU(approximate='none')
    (4): Conv2d(96, 192, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (5): GELU(approximate='none')
  )
  (blocks): ModuleList(
    (0-3): 4 x DHVTBlock(
      (norm1): LayerNorm((192,), eps=1e-05, elementwise_affine=True)
      (attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=192, out_features=192, bias=True)
      )
      (norm2): LayerNorm((192,), eps=1e-05, elementwise_affine=True)
      (mlp): ConvMLP(
        (fc1): Linear(in_features=192, out_features=768, bias=True)
        (act): GELU(approximate='none')
        (dwconv): Conv2d(768, 768, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=768)
        (fc2): Linear(in_features=768, out_features=192, bias=Tru

**Pro tip.** Pytorch lighting proposes many very convenient flags to try and debug before testing. For instance, `fast_dev_run` runs some training and validation steps in order to check that everyting works.

## Train for some epochs


In [12]:
MAX_EPOCHS = 100

trainer = pl.Trainer(max_epochs=MAX_EPOCHS,
                     accelerator="auto",
                     devices="auto")

trainer.fit(net, train_dataloaders=train_loader, val_dataloaders=valid_loader)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
c:\Users\Yvan\Desktop\UPC\Deep Learning\TP\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\logger_connector\logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpo

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\Users\Yvan\Desktop\UPC\Deep Learning\TP\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Epoch 99: 100%|██████████| 55/55 [00:03<00:00, 14.82it/s, v_num=44, valid_loss=2.420, valid_acc=0.547, train_loss=0.154, train_acc=0.946, learning_rate=0.0001]

`Trainer.fit` stopped: `max_epochs=100` reached.


Epoch 99: 100%|██████████| 55/55 [00:03<00:00, 14.18it/s, v_num=44, valid_loss=2.420, valid_acc=0.547, train_loss=0.154, train_acc=0.946, learning_rate=0.0001]


In [13]:
trainer.save_checkpoint("ViT_modele_" + DATA + ".ckpt")

`weights_only` was not set, defaulting to `False`.


In [14]:
from torchmetrics.classification import MulticlassConfusionMatrix
import torch

metric = MulticlassConfusionMatrix(num_classes=num_classes)

if DATA in ("all", "texture"):
    test_dataset_texture = datasets.ImageFolder(
        root="./datasets_split/test/texture",
        transform=transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor()
    ])
    )

    test_loader_texture = DataLoader(test_dataset_texture, batch_size=64, shuffle=True, num_workers=num_workers, persistent_workers=True, pin_memory=True)

    net.eval()
    with torch.no_grad():
        for x, y in test_loader_texture:

            logits = net(x)
            preds = torch.argmax(logits, dim=1)
            metric.update(preds, y)

    cm_text = metric.compute()
    metric.reset()



if DATA in ("all", "long_range"):
    test_dataset_long_range = datasets.ImageFolder(
        root="./datasets_split/test/long_range",
        transform=transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor()
    ])
    )

    test_loader_long_range = DataLoader(test_dataset_long_range, batch_size=64, shuffle=True, num_workers=num_workers, persistent_workers=True, pin_memory=True)

    with torch.no_grad():
        for x, y in test_loader_long_range:

            logits = net(x)
            preds = torch.argmax(logits, dim=1)
            metric.update(preds, y)

    cm_long = metric.compute()


In [15]:
data = {}

# Texture
if DATA in ("all", "texture"):
    data["texture_confusion_matrix"] = cm_text.tolist()
    cm_text

In [16]:
# Long range
if DATA in ("all", "long_range"):
    data["long_range_confusion_matrix"] = cm_long.tolist()
    cm_long

In [17]:

with open("test_confusion_matrix_" +DATA+ ".json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2)